# 5. MODEL EVALUATION AND DIAGNOSTICS
## Daily Customer Churn Predictor · VivaMarket Brasil

---

**INPUT:** `../models/churn_model_YYYYMMDD.joblib`, `../data/processed/churn_features_YYYYMMDD.parquet`, and `../data/processed/churn_predictions_YYYYMMDD.parquet`

*The selected churn model, the full feature matrix, and the scored test snapshots produced in NB04.*

**OUTPUT:** `../data/processed/churn_diagnostics_YYYYMMDD.csv` and `../reports/model_diagnostics_YYYYMMDD.html`

*A business-facing diagnostic package covering ranking quality, calibration, temporal stability, and threshold trade-offs.*


---
## 5.1. STARTING SITUATION


NB04 selected the best-performing model under a temporal split and produced a scored test population with operational risk tiers. Before moving into explainability and deployment, the project needs a more rigorous view of how reliable those scores really are.

This notebook therefore turns raw predictive output into a **decision-quality diagnostic layer**. The goal is to verify whether the model remains useful across future monthly snapshots, how concentrated risk is at the top of the ranking, and how calibration and threshold choices affect the retention workload.


---
## 5.2. NOTEBOOK OBJECTIVE


- **Business objective:** verify that the selected churn model prioritizes the right customers for retention actions and supports economically reasonable contact thresholds.
- **Analytical objective:** quantify ranking quality, calibration, temporal stability, and campaign concentration across the held-out test period.


---
## 5.3. INITIAL SETUP

**What is done**

We load the libraries required for diagnostics, plotting, model loading, and HTML report generation.

**Why it is done**

NB05 must be reproducible and explicit because the evaluation stage is where modeling quality becomes a business decision.

**Expected result**

A stable environment with resolved paths, active logging, and report folders ready for diagnostic outputs.


In [1]:
import base64
import io
import logging
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve, roc_auc_score, roc_curve

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', force=True)
logger = logging.getLogger('nb05_model_evaluation')
logger.info('NB05 started: model evaluation and diagnostics.')

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


2026-05-02 00:38:26,821 | INFO | NB05 started: model evaluation and diagnostics.


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

run_date_tag = datetime.now(ZoneInfo('Europe/Paris')).strftime('%Y%m%d')
model_path = sorted(MODELS_DIR.glob('churn_model_*.joblib'))[-1]
feature_path = sorted(PROCESSED_DIR.glob('churn_features_*.parquet'))[-1]
prediction_path = sorted(PROCESSED_DIR.glob('churn_predictions_*.parquet'))[-1]
diagnostics_csv_path = PROCESSED_DIR / f'churn_diagnostics_{run_date_tag}.csv'
diagnostics_html_path = REPORTS_DIR / f'model_diagnostics_{run_date_tag}.html'

logger.info('Model path: %s', model_path)
logger.info('Feature path: %s', feature_path)
logger.info('Prediction path: %s', prediction_path)


2026-05-02 00:38:26,829 | INFO | Model path: /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/models/churn_model_20260502.joblib


2026-05-02 00:38:26,829 | INFO | Feature path: /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_features_20260502.parquet


2026-05-02 00:38:26,830 | INFO | Prediction path: /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_predictions_20260502.parquet


---
## 5.4. DATA RECONSTRUCTION FOR DIAGNOSTICS

**What is done**

We reload the model package, rebuild the same encoded feature space used in NB04, and align the test rows with the persisted scored output.

**Why it is done**

Deep diagnostics need access to both the full feature matrix and the production-like prediction file so that ranking, stability and threshold analysis remain fully auditable.

**Expected result**

A matched diagnostic table for the held-out test period, including probabilities, labels, snapshot dates and campaign tiers.


In [3]:
package = joblib.load(model_path)
feature_df = pd.read_parquet(feature_path)
feature_df['snapshot_date'] = pd.to_datetime(feature_df['snapshot_date'])
prediction_df = pd.read_parquet(prediction_path)
prediction_df['snapshot_date'] = pd.to_datetime(prediction_df['snapshot_date'])

test_keys = package['test_snapshot_keys']
feature_columns = package['feature_columns']
scored_model = package['model']

leakage_columns = [
    'customer_unique_id', 'snapshot_key', 'snapshot_date', 'first_purchase_timestamp',
    'last_purchase_timestamp', 'future_orders_90d', 'future_revenue_90d', 'churn_90d_label'
]
test_df = feature_df[feature_df['snapshot_key'].isin(test_keys)].copy()
X_test = pd.get_dummies(
    test_df[[c for c in feature_df.columns if c not in leakage_columns]],
    columns=['customer_state'],
    dtype=float,
)
X_test = X_test.reindex(columns=feature_columns, fill_value=0.0)
test_df['recomputed_probability'] = scored_model.predict_proba(X_test)[:, 1]

diagnostics_df = prediction_df.merge(
    test_df[[
        'customer_unique_id', 'snapshot_key', 'snapshot_date', 'future_orders_90d',
        'future_revenue_90d', 'recomputed_probability'
    ]],
    on=['customer_unique_id', 'snapshot_key', 'snapshot_date'],
    how='left',
    validate='one_to_one',
)
diagnostics_df['probability_diff'] = diagnostics_df['churn_probability'] - diagnostics_df['recomputed_probability']
logger.info('Maximum scoring reconstruction difference: %.10f', diagnostics_df['probability_diff'].abs().max())
diagnostics_df.head()


2026-05-02 00:38:27,389 | INFO | Maximum scoring reconstruction difference: 0.0000000000


,customer_unique_id,snapshot_key,snapshot_date,recency_days,total_orders,total_payment_value,orders_30d,orders_90d,churn_90d_label,churn_probability,risk_tier,selected_model,future_orders_90d,future_revenue_90d,recomputed_probability,probability_diff
0,0004bd2a26a76fe21f786e4fbd80607f,20180501,2018-05-01,26,1,166.9800,1.0000,1,1,0.5585,MEDIUM,xgboost,0.0000,0.0000,0.5585,0.0000
1,00050ab1314c0e55a6ca13cf7181fecf,20180501,2018-05-01,11,1,35.3800,1.0000,1,1,0.5727,MEDIUM,xgboost,0.0000,0.0000,0.5727,0.0000
2,00053a61a98854899e70ed204dd4bafe,20180501,2018-05-01,62,1,419.1800,0.0000,1,1,0.8962,HIGH,xgboost,0.0000,0.0000,0.8962,0.0000
3,0005ef4cd20d2893f0d9fbd94d3c0d97,20180501,2018-05-01,50,1,129.7600,0.0000,1,1,0.7706,HIGH,xgboost,0.0000,0.0000,0.7706,0.0000
4,00090324bbad0e9342388303bb71ba0a,20180501,2018-05-01,38,1,63.6600,0.0000,1,1,0.7081,HIGH,xgboost,0.0000,0.0000,0.7081,0.0000


---
## 5.5. GLOBAL PERFORMANCE AND THRESHOLD TRADE-OFFS

**What is done**

We quantify overall ranking quality, threshold trade-offs, and campaign concentration at the top of the score distribution.

**Why it is done**

Retention budgets care about who appears first in the ranking, how many customers would be contacted, and what observed churn rate sits inside each operational slice.

**Expected result**

A concise metric package that links predictive performance to the High / Medium / Low retention framework.


In [4]:
def precision_at_top_fraction(y_true: pd.Series, scores: pd.Series, fraction: float) -> float:
    rank_df = pd.DataFrame({'y_true': y_true.to_numpy(), 'score': scores.to_numpy()})
    rank_df = rank_df.sort_values('score', ascending=False).reset_index(drop=True)
    cutoff = max(int(np.ceil(len(rank_df) * fraction)), 1)
    return float(rank_df.head(cutoff)['y_true'].mean())

y_true = diagnostics_df['churn_90d_label'].astype(int)
y_score = diagnostics_df['churn_probability'].astype(float)

metric_rows = [
    ('roc_auc', roc_auc_score(y_true, y_score)),
    ('average_precision', average_precision_score(y_true, y_score)),
    ('brier_score', brier_score_loss(y_true, y_score)),
    ('precision_at_top_1pct', precision_at_top_fraction(y_true, y_score, 0.01)),
    ('precision_at_top_5pct', precision_at_top_fraction(y_true, y_score, 0.05)),
    ('precision_at_top_10pct', precision_at_top_fraction(y_true, y_score, 0.10)),
    ('high_risk_share', float((diagnostics_df['risk_tier'] == 'HIGH').mean())),
    ('medium_risk_share', float((diagnostics_df['risk_tier'] == 'MEDIUM').mean())),
    ('low_risk_share', float((diagnostics_df['risk_tier'] == 'LOW').mean())),
]
summary_metrics = pd.DataFrame(metric_rows, columns=['metric', 'value'])
summary_metrics


,metric,value
0,roc_auc,0.5888
1,average_precision,0.9937
2,brier_score,0.1600
3,precision_at_top_1pct,0.9934
4,precision_at_top_5pct,0.9921
5,precision_at_top_10pct,0.9934
6,high_risk_share,0.3081
7,medium_risk_share,0.6336
8,low_risk_share,0.0583


In [5]:
threshold_grid = np.round(np.arange(0.30, 0.91, 0.05), 2)
threshold_rows = []
for threshold in threshold_grid:
    targeted = diagnostics_df['churn_probability'] >= threshold
    contacts = int(targeted.sum())
    threshold_rows.append({
        'threshold': threshold,
        'targeted_rows': contacts,
        'targeted_share': float(targeted.mean()),
        'observed_churn_rate': float(diagnostics_df.loc[targeted, 'churn_90d_label'].mean()) if contacts else np.nan,
        'avg_future_revenue_90d': float(diagnostics_df.loc[targeted, 'future_revenue_90d'].mean()) if contacts else np.nan,
    })
threshold_df = pd.DataFrame(threshold_rows)
threshold_df


,threshold,targeted_rows,targeted_share,observed_churn_rate,avg_future_revenue_90d
0,0.3000,59266,0.9809,0.9931,1.0729
1,0.3500,58409,0.9667,0.9932,1.0563
2,0.4000,56897,0.9417,0.9932,1.0568
3,0.4500,54315,0.8989,0.9934,1.0334
4,0.5000,50169,0.8303,0.9936,0.9805
5,0.5500,44053,0.7291,0.9939,0.9290
6,0.6000,36345,0.6015,0.9938,0.9915
7,0.6500,27426,0.4539,0.9941,1.0205
8,0.7000,18613,0.3081,0.9941,1.0523
9,0.7500,11078,0.1833,0.9938,1.2182


---
## 5.6. TEMPORAL STABILITY AND CALIBRATION

**What is done**

We evaluate diagnostics by monthly snapshot and compare predicted scores with observed churn rates through calibration bins.

**Why it is done**

A model that looks good in aggregate can still drift month to month or overstate confidence in certain parts of the score distribution.

**Expected result**

A stable monthly view that reveals whether ranking quality and confidence remain usable for daily campaign operations.


In [6]:
monthly_backtest = (
    diagnostics_df.groupby('snapshot_key', observed=True)
    .apply(lambda frame: pd.Series({
        'rows_n': len(frame),
        'observed_churn_rate': frame['churn_90d_label'].mean(),
        'avg_score': frame['churn_probability'].mean(),
        'precision_at_top_10pct': precision_at_top_fraction(frame['churn_90d_label'].astype(int), frame['churn_probability'], 0.10),
        'average_precision': average_precision_score(frame['churn_90d_label'].astype(int), frame['churn_probability']),
    }), include_groups=False)
    .reset_index()
)
monthly_backtest


,snapshot_key,rows_n,observed_churn_rate,avg_score,precision_at_top_10pct,average_precision
0,20180501,"20,726.0000",0.9907,0.6289,0.9899,0.9921
1,20180601,"20,183.0000",0.9920,0.6261,0.9941,0.9937
2,20180701,"19,512.0000",0.9946,0.6208,0.9969,0.9956


In [7]:
calibration_bins = pd.qcut(diagnostics_df['churn_probability'], q=10, duplicates='drop')
calibration_df = (
    diagnostics_df.assign(calibration_bin=calibration_bins)
    .groupby('calibration_bin', observed=True)
    .agg(
        customers=('customer_unique_id', 'nunique'),
        avg_predicted_probability=('churn_probability', 'mean'),
        observed_churn_rate=('churn_90d_label', 'mean'),
    )
    .reset_index()
)
calibration_df


,calibration_bin,customers,avg_predicted_probability,observed_churn_rate
0,"(0.047599999999999996, 0.449]",4482,0.3617,0.9835
1,"(0.449, 0.516]",5331,0.4860,0.9911
2,"(0.516, 0.563]",5453,0.5404,0.9924
3,"(0.563, 0.601]",5503,0.5819,0.9940
4,"(0.601, 0.635]",5469,0.6180,0.9922
5,"(0.635, 0.668]",5484,0.6515,0.9942
6,"(0.668, 0.703]",5452,0.6852,0.9934
7,"(0.703, 0.742]",5339,0.7222,0.9952
8,"(0.742, 0.794]",5146,0.7667,0.9944
9,"(0.794, 0.982]",4379,0.8400,0.9934


---
## 5.7. DIAGNOSTIC VISUALS AND HTML REPORT


In [8]:
def figure_to_base64(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', bbox_inches='tight', dpi=160)
    plt.close(fig)
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

roc_fpr, roc_tpr, _ = roc_curve(y_true, y_score)
pr_precision, pr_recall, _ = precision_recall_curve(y_true, y_score)
prob_true, prob_pred = calibration_curve(y_true, y_score, n_bins=10, strategy='quantile')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].plot(roc_fpr, roc_tpr, label=f'ROC AUC = {roc_auc_score(y_true, y_score):.3f}', color='#1f77b4')
axes[0, 0].plot([0, 1], [0, 1], linestyle='--', color='grey')
axes[0, 0].set_title('ROC CURVE')
axes[0, 0].legend()

axes[0, 1].plot(pr_recall, pr_precision, color='#ff7f0e')
axes[0, 1].set_title('PRECISION-RECALL CURVE')
axes[0, 1].set_xlabel('Recall')
axes[0, 1].set_ylabel('Precision')

sns.lineplot(data=monthly_backtest, x='snapshot_key', y='precision_at_top_10pct', marker='o', ax=axes[1, 0])
axes[1, 0].set_title('MONTHLY PRECISION AT TOP 10%')
axes[1, 0].tick_params(axis='x', rotation=45)

axes[1, 1].plot(prob_pred, prob_true, marker='o', color='#2ca02c')
axes[1, 1].plot([0, 1], [0, 1], linestyle='--', color='grey')
axes[1, 1].set_title('CALIBRATION CURVE')
axes[1, 1].set_xlabel('Predicted probability')
axes[1, 1].set_ylabel('Observed churn rate')

plt.tight_layout()
diagnostics_chart = figure_to_base64(fig)

risk_summary = (
    diagnostics_df.groupby('risk_tier', observed=False)
    .agg(rows_n=('customer_unique_id', 'size'), customers_n=('customer_unique_id', 'nunique'), observed_churn_rate=('churn_90d_label', 'mean'), avg_probability=('churn_probability', 'mean'))
    .reset_index()
)

html_parts = [
    '<html><head><meta charset="utf-8"><title>Model Diagnostics</title></head><body>',
    '<h1>MODEL DIAGNOSTICS REPORT</h1>',
    '<h2>Summary metrics</h2>', summary_metrics.to_html(index=False),
    '<h2>Risk-tier summary</h2>', risk_summary.to_html(index=False),
    '<h2>Threshold trade-offs</h2>', threshold_df.to_html(index=False),
    '<h2>Monthly backtest</h2>', monthly_backtest.to_html(index=False),
    '<h2>Calibration bins</h2>', calibration_df.to_html(index=False),
    f'<h2>Diagnostic visuals</h2><img src="data:image/png;base64,{diagnostics_chart}" style="max-width:1100px;">',
    '</body></html>'
]
diagnostics_html_path.write_text('\n'.join(html_parts), encoding='utf-8')

diagnostics_export = pd.concat([
    summary_metrics.assign(section='summary'),
    threshold_df.assign(section='thresholds').rename(columns={'threshold': 'metric', 'targeted_share': 'value'}),
], ignore_index=True, sort=False)
diagnostics_export.to_csv(diagnostics_csv_path, index=False)
logger.info('Diagnostics CSV saved to %s', diagnostics_csv_path)
logger.info('Diagnostics HTML report saved to %s', diagnostics_html_path)


2026-05-02 00:38:27,602 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


2026-05-02 00:38:27,607 | INFO | Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


2026-05-02 00:38:28,114 | INFO | Diagnostics CSV saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_diagnostics_20260502.csv


2026-05-02 00:38:28,115 | INFO | Diagnostics HTML report saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/reports/model_diagnostics_20260502.html


---
## 5.8. NOTEBOOK CLOSURE


The diagnostic stage confirms whether the selected ranking is stable enough to drive retention actions. The main operational value of this notebook is that it reframes model quality as **campaign quality**: who gets contacted, how much churn concentrates in the top ranks, and how stable the score remains across future monthly snapshots.

The next notebook should explain *why* those customers score high risk by translating the model into SHAP-based churn drivers and segment-level narratives.
